# Módulo Geoespacial — Capa técnica transversal

**Bloque C — Técnicas transversales** | Módulo: `spatial/`

> Este módulo es la **capa transversal** que da soporte espacial a todas las líneas del repositorio:
> áreas protegidas, páramos, humedales, calidad del aire, oferta hídrica, deforestación, gestión del riesgo.

---

## Casos de uso en este notebook

| # | Pregunta analítica | Método | Función |
|---|---|---|---|
| 1 | ¿Cuánta área de una iniciativa REDD+ está dentro de un área protegida? | Overlay poligonal | `intersection_area()` |
| 2 | ¿Qué emisiones GEI corresponden a cada departamento? | Estadísticas zonales | `zonal_statistics()` |
| 3 | ¿Cómo estimar temperatura en municipios sin estación IDEAM? | IDW · Kriging ordinario · Universal | `idw()`, `ordinary_kriging()`, `universal_kriging()` |
| 4 | ¿Dónde se concentran espacialmente los focos de deforestación? | Autocorrelación espacial | `morans_i()`, `geary_c()`, `getis_ord_g()` |

**Dependencias:** `pip install estadistica-ambiental[spatial]`  
Incluye: `geopandas`, `rasterio`, `pykrige`, `pysal`, `esda`, `folium`, `shapely`

*Retroalimentación metodológica: Juan (SIG — Ministerio de Ambiente y Desarrollo Sostenible)*

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Módulo spatial/ del repositorio
from estadistica_ambiental.spatial.analysis import intersection_area, zonal_statistics
from estadistica_ambiental.spatial.interpolation import idw, ordinary_kriging, universal_kriging
from estadistica_ambiental.spatial.autocorrelation import morans_i, geary_c, getis_ord_g, local_morans_i
from estadistica_ambiental.spatial.projections import points_to_geodataframe, clip_to_colombia
from estadistica_ambiental.spatial.viz import map_stations, choropleth_map, plot_kriging_map

# Dependencias opcionales [spatial]
try:
    import geopandas as gpd
    from shapely.geometry import box, Point
    HAS_GPD = True
    print(f'geopandas {gpd.__version__} OK')
except ImportError:
    HAS_GPD = False
    print('AVISO: pip install estadistica-ambiental[spatial] para ejecutar este notebook')

try:
    import rasterio
    from rasterio.transform import from_bounds
    HAS_RASTERIO = True
except ImportError:
    HAS_RASTERIO = False

try:
    import pykrige
    HAS_PYKRIGE = True
except ImportError:
    HAS_PYKRIGE = False

try:
    import esda
    HAS_ESDA = True
except ImportError:
    HAS_ESDA = False

DOCS = Path('../../..') / 'docs'

<!-- ENRICHMENT: geoespacial -->

---

## Caso 5 — NDWI y MNDWI: Deteccion de espejo de agua con Sentinel-2

Los indices de agua permiten extraer cuerpos de agua de imagenes multiespectrales sin clasificacion supervisada:

| Indice | Formula | Bandas Sentinel-2 | Umbral agua |
|---|---|---|---|
| **NDWI** (McFeeters) | (Green - NIR) / (Green + NIR) | B3 / B8 | > 0 |
| **MNDWI** (Xu 2006) | (Green - SWIR) / (Green + SWIR) | B3 / B11 | > 0 |
| **AWEI** | 4*(Green-SWIR1) - (0.25*NIR+2.75*SWIR2) | B3/B11/B8/B12 | > 0 |

**NDWI** es sensible a la turbidez; **MNDWI** es mas robusto en zonas urbanas y humedales someros porque suprime mejor la respuesta de suelo desnudo y edificios.

> En humedales colombianos: usar MNDWI sobre Sentinel-2 L2A (correccion atmosferica), mascara de nubes SCL (escena de clasificacion de escena) y combinacion con Sentinel-1 SAR para periodos nublados (Amazonia/Pacifico).

In [ ]:
# NDWI y MNDWI simulados: comparacion de indices de agua
np.random.seed(42)

# Simular una imagen de 100x100 pixeles con distintas coberturas
n_pix = 10000
# Reflectancias Sentinel-2 L2A por cobertura (valores tipicos 0-1)
coberturas = {
    'Agua profunda':   {'green': 0.03, 'nir': 0.01, 'swir1': 0.005},
    'Agua turbia':     {'green': 0.06, 'nir': 0.04, 'swir1': 0.03},
    'Vegetacion':      {'green': 0.10, 'nir': 0.35, 'swir1': 0.15},
    'Suelo desnudo':   {'green': 0.15, 'nir': 0.22, 'swir1': 0.20},
    'Area urbana':     {'green': 0.18, 'nir': 0.20, 'swir1': 0.18},
    'Nieve/hielo':     {'green': 0.45, 'nir': 0.40, 'swir1': 0.10},
}

n_por_cob = n_pix // len(coberturas)
all_green, all_nir, all_swir1, all_labels = [], [], [], []
for nombre, vals in coberturas.items():
    noise = np.random.normal(0, 0.005, n_por_cob)
    all_green.append(np.clip(vals['green'] + noise, 0, 1))
    all_nir.append(np.clip(vals['nir'] + noise, 0, 1))
    all_swir1.append(np.clip(vals['swir1'] + noise, 0, 1))
    all_labels.extend([nombre] * n_por_cob)

green = np.concatenate(all_green)
nir   = np.concatenate(all_nir)
swir1 = np.concatenate(all_swir1)

ndwi  = (green - nir)   / (green + nir  + 1e-8)   # NDWI McFeeters
mndwi = (green - swir1) / (green + swir1 + 1e-8)  # MNDWI Xu

df_indices = pd.DataFrame({
    'cobertura': all_labels, 'green': green, 'nir': nir,
    'swir1': swir1, 'ndwi': ndwi, 'mndwi': mndwi
})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel A: Distribucion NDWI por cobertura
ax = axes[0]
for cob in coberturas.keys():
    sub = df_indices[df_indices['cobertura'] == cob]['ndwi']
    ax.boxplot(sub, positions=[list(coberturas).index(cob)],
              widths=0.6, patch_artist=True,
              boxprops=dict(facecolor='#3498db', alpha=0.6))
ax.axhline(0, color='red', ls='--', lw=1.5, label='Umbral agua NDWI=0')
ax.set_xticks(range(len(coberturas)))
ax.set_xticklabels(coberturas.keys(), rotation=30, ha='right', fontsize=8)
ax.set_title('NDWI (McFeeters) por cobertura', fontweight='bold')
ax.set_ylabel('NDWI'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel B: Comparacion NDWI vs MNDWI
ax = axes[1]
colors_cob = ['#1a73e8','#27ae60','#e67e22','#e74c3c','#9b59b6','#f1c40f']
for cob, color in zip(coberturas.keys(), colors_cob):
    sub = df_indices[df_indices['cobertura'] == cob]
    ax.scatter(sub['ndwi'].sample(100, random_state=1), sub['mndwi'].sample(100, random_state=1),
               c=color, alpha=0.5, s=15, label=cob)
ax.axhline(0, color='gray', ls='--', lw=1); ax.axvline(0, color='gray', ls='--', lw=1)
ax.set_xlabel('NDWI'); ax.set_ylabel('MNDWI')
ax.set_title('NDWI vs MNDWI — separacion coberturas\n(Sentinel-2, CTM12 EPSG:9377)', fontweight='bold')
ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.3)

# Panel C: Precision deteccion agua (NDWI vs MNDWI)
ax = axes[2]
es_agua = df_indices['cobertura'].str.contains('Agua').astype(int)
ndwi_agua = (ndwi > 0).astype(int)
mndwi_agua = (mndwi > 0).astype(int)
prec_ndwi  = (ndwi_agua == es_agua).mean()
prec_mndwi = (mndwi_agua == es_agua).mean()
ax.bar(['NDWI\n(McFeeters)','MNDWI\n(Xu 2006)'], [prec_ndwi, prec_mndwi],
       color=['#3498db','#27ae60'], alpha=0.85, width=0.5)
ax.set_ylim(0, 1); ax.set_ylabel('Exactitud global')
ax.set_title('Precision deteccion agua\nNDWI vs MNDWI (umbral=0)', fontweight='bold')
for i, v in enumerate([prec_ndwi, prec_mndwi]):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('NDWI/MNDWI — Extraccion espejo de agua Sentinel-2 (EPSG:9377 para areas)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()
print(f'Exactitud NDWI: {prec_ndwi:.3f} | MNDWI: {prec_mndwi:.3f}')
print('MNDWI recomendado en humedales urbanos y turbios (menor confusion suelo/agua)')

---

## Caso 5 — NDWI y MNDWI: Deteccion de espejo de agua con Sentinel-2

Los indices de agua permiten extraer cuerpos de agua de imagenes multiespectrales sin clasificacion supervisada:

| Indice | Formula | Bandas Sentinel-2 | Umbral agua |
|---|---|---|---|
| **NDWI** (McFeeters) | (Green - NIR) / (Green + NIR) | B3 / B8 | > 0 |
| **MNDWI** (Xu 2006) | (Green - SWIR) / (Green + SWIR) | B3 / B11 | > 0 |
| **AWEI** | 4*(Green-SWIR1) - (0.25*NIR+2.75*SWIR2) | B3/B11/B8/B12 | > 0 |

**NDWI** es sensible a la turbidez; **MNDWI** es mas robusto en zonas urbanas y humedales someros porque suprime mejor la respuesta de suelo desnudo y edificios.

> En humedales colombianos: usar MNDWI sobre Sentinel-2 L2A (correccion atmosferica), mascara de nubes SCL (escena de clasificacion de escena) y combinacion con Sentinel-1 SAR para periodos nublados (Amazonia/Pacifico).

In [ ]:
# NDWI y MNDWI simulados: comparacion de indices de agua
np.random.seed(42)

# Simular una imagen de 100x100 pixeles con distintas coberturas
n_pix = 10000
# Reflectancias Sentinel-2 L2A por cobertura (valores tipicos 0-1)
coberturas = {
    'Agua profunda':   {'green': 0.03, 'nir': 0.01, 'swir1': 0.005},
    'Agua turbia':     {'green': 0.06, 'nir': 0.04, 'swir1': 0.03},
    'Vegetacion':      {'green': 0.10, 'nir': 0.35, 'swir1': 0.15},
    'Suelo desnudo':   {'green': 0.15, 'nir': 0.22, 'swir1': 0.20},
    'Area urbana':     {'green': 0.18, 'nir': 0.20, 'swir1': 0.18},
    'Nieve/hielo':     {'green': 0.45, 'nir': 0.40, 'swir1': 0.10},
}

n_por_cob = n_pix // len(coberturas)
all_green, all_nir, all_swir1, all_labels = [], [], [], []
for nombre, vals in coberturas.items():
    noise = np.random.normal(0, 0.005, n_por_cob)
    all_green.append(np.clip(vals['green'] + noise, 0, 1))
    all_nir.append(np.clip(vals['nir'] + noise, 0, 1))
    all_swir1.append(np.clip(vals['swir1'] + noise, 0, 1))
    all_labels.extend([nombre] * n_por_cob)

green = np.concatenate(all_green)
nir   = np.concatenate(all_nir)
swir1 = np.concatenate(all_swir1)

ndwi  = (green - nir)   / (green + nir  + 1e-8)   # NDWI McFeeters
mndwi = (green - swir1) / (green + swir1 + 1e-8)  # MNDWI Xu

df_indices = pd.DataFrame({
    'cobertura': all_labels, 'green': green, 'nir': nir,
    'swir1': swir1, 'ndwi': ndwi, 'mndwi': mndwi
})

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Panel A: Distribucion NDWI por cobertura
ax = axes[0]
for cob in coberturas.keys():
    sub = df_indices[df_indices['cobertura'] == cob]['ndwi']
    ax.boxplot(sub, positions=[list(coberturas).index(cob)],
              widths=0.6, patch_artist=True,
              boxprops=dict(facecolor='#3498db', alpha=0.6))
ax.axhline(0, color='red', ls='--', lw=1.5, label='Umbral agua NDWI=0')
ax.set_xticks(range(len(coberturas)))
ax.set_xticklabels(coberturas.keys(), rotation=30, ha='right', fontsize=8)
ax.set_title('NDWI (McFeeters) por cobertura', fontweight='bold')
ax.set_ylabel('NDWI'); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Panel B: Comparacion NDWI vs MNDWI
ax = axes[1]
colors_cob = ['#1a73e8','#27ae60','#e67e22','#e74c3c','#9b59b6','#f1c40f']
for cob, color in zip(coberturas.keys(), colors_cob):
    sub = df_indices[df_indices['cobertura'] == cob]
    ax.scatter(sub['ndwi'].sample(100, random_state=1), sub['mndwi'].sample(100, random_state=1),
               c=color, alpha=0.5, s=15, label=cob)
ax.axhline(0, color='gray', ls='--', lw=1); ax.axvline(0, color='gray', ls='--', lw=1)
ax.set_xlabel('NDWI'); ax.set_ylabel('MNDWI')
ax.set_title('NDWI vs MNDWI — separacion coberturas\n(Sentinel-2, CTM12 EPSG:9377)', fontweight='bold')
ax.legend(fontsize=6, ncol=2); ax.grid(alpha=0.3)

# Panel C: Precision deteccion agua (NDWI vs MNDWI)
ax = axes[2]
es_agua = df_indices['cobertura'].str.contains('Agua').astype(int)
ndwi_agua = (ndwi > 0).astype(int)
mndwi_agua = (mndwi > 0).astype(int)
prec_ndwi  = (ndwi_agua == es_agua).mean()
prec_mndwi = (mndwi_agua == es_agua).mean()
ax.bar(['NDWI\n(McFeeters)','MNDWI\n(Xu 2006)'], [prec_ndwi, prec_mndwi],
       color=['#3498db','#27ae60'], alpha=0.85, width=0.5)
ax.set_ylim(0, 1); ax.set_ylabel('Exactitud global')
ax.set_title('Precision deteccion agua\nNDWI vs MNDWI (umbral=0)', fontweight='bold')
for i, v in enumerate([prec_ndwi, prec_mndwi]):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center', fontweight='bold')
ax.grid(axis='y', alpha=0.3)

plt.suptitle('NDWI/MNDWI — Extraccion espejo de agua Sentinel-2 (EPSG:9377 para areas)',
             fontweight='bold', fontsize=11)
plt.tight_layout(); plt.show()
print(f'Exactitud NDWI: {prec_ndwi:.3f} | MNDWI: {prec_mndwi:.3f}')
print('MNDWI recomendado en humedales urbanos y turbios (menor confusion suelo/agua)')

## 0b. Contexto de dominio

In [ ]:
ficha = DOCS / 'fuentes' / 'geoespacial.md'
if ficha.exists():
    print(ficha.read_text(encoding='utf-8')[:2500])

---

## Caso 1 — Traslapes poligonales: REDD+ ↔ Áreas Protegidas

### Contexto

Las iniciativas de créditos de carbono (REDD+, proyectos forestales voluntarios) deben verificar
qué fracción de su área se superpone con figuras de conservación existentes (Parques Nacionales,
Reservas Forestales, páramos delimitados). Sin esta verificación, los créditos pueden registrarse
sobre territorio ya protegido, comprometiendo la **adicionalidad** del proyecto.

### Fuentes de datos reales

| Capa | Fuente | Dónde obtener |
|---|---|---|
| Iniciativas REDD+ | RENARE (MADS) — solicitud institucional | Shapefile por proyecto |
| Áreas protegidas | RUNAP — SIAC | https://www.siac.gov.co/runap |
| Páramos delimitados | IAvH / IDEAM | https://www.humboldt.org.co |
| Límites departamentales | IGAC | https://geoportal.igac.gov.co |

### Decisión técnica clave

> `intersection_area()` reprojecta automáticamente a **CTM12 (EPSG:9377)** antes de calcular áreas.
> Calcular áreas en WGS84 (grados decimales) introduce errores de hasta 30% en Colombia.
> Ver ADR-006 en `docs/decisiones.md`.

In [ ]:
if not HAS_GPD:
    print('Requiere: pip install estadistica-ambiental[spatial]')
else:
    # Iniciativas REDD+ sintéticas — Cundinamarca / Boyacá
    redd = gpd.GeoDataFrame(
        {
            'id_redd':  ['REDD-001', 'REDD-002', 'REDD-003', 'REDD-004'],
            'nombre':   ['Bosque Andino Norte', 'Corredor Andes-Orinoquia',
                         'Reserva Campesina Sur', 'Páramo Sumapaz Buffer'],
            'ha_total': [12500, 8300, 5100, 9800],
            'geometry': [
                box(-74.20, 4.60, -73.90, 4.85),
                box(-73.50, 4.20, -73.10, 4.55),
                box(-74.50, 3.80, -74.20, 4.10),
                box(-74.35, 4.10, -74.05, 4.45),
            ],
        },
        crs='EPSG:4326',
    )

    # Áreas protegidas sintéticas
    ap = gpd.GeoDataFrame(
        {
            'id_ap':  ['PNN-Chingaza', 'RF-Andes-Sur', 'DMI-Sumapaz'],
            'figura': ['Parque Nacional Natural', 'Reserva Forestal', 'Distrito de Manejo Integrado'],
            'geometry': [
                box(-74.30, 4.50, -73.95, 4.80),
                box(-73.60, 4.10, -73.20, 4.50),
                box(-74.40, 4.05, -74.10, 4.40),
            ],
        },
        crs='EPSG:4326',
    )

    print(f'Iniciativas REDD+: {len(redd)} polígonos')
    print(f'Áreas protegidas:  {len(ap)} polígonos')
    print()
    print(redd[['id_redd', 'nombre', 'ha_total']])

In [ ]:
if HAS_GPD:
    # Calcular traslapes — usa overlay vectorizado + CTM12 para áreas en m²
    traslapes = intersection_area(redd, ap, id_col1='id_redd', id_col2='id_ap')
    traslapes['ha_traslape'] = (traslapes['intersection_area_m2'] / 10_000).round(1)

    print(f'Traslapes encontrados: {len(traslapes)}\n')
    cols = ['id_redd', 'id_ap', 'ha_traslape', 'pct_of_id_redd', 'pct_of_id_ap']
    print(traslapes[cols].round(2).to_string(index=False))

In [ ]:
if HAS_GPD:
    fig, ax = plt.subplots(figsize=(10, 8))

    redd.plot(ax=ax, color='#2ecc71', alpha=0.4, edgecolor='#27ae60', linewidth=1.5, label='Iniciativas REDD+')
    ap.plot(ax=ax, color='#3498db', alpha=0.35, edgecolor='#2980b9', linewidth=1.5, label='Áreas protegidas')
    if not traslapes.empty:
        traslapes.plot(ax=ax, color='#e74c3c', alpha=0.75, edgecolor='#c0392b', linewidth=1, label='Traslape')

    for _, row in redd.iterrows():
        ax.annotate(row['id_redd'], (row.geometry.centroid.x, row.geometry.centroid.y),
                    ha='center', fontsize=8, color='#1a5276', fontweight='bold')
    for _, row in ap.iterrows():
        ax.annotate(row['figura'].split()[0], (row.geometry.centroid.x, row.geometry.centroid.y),
                    ha='center', fontsize=7, color='#1a5276')

    ax.set_title('Traslapes REDD+ ↔ Áreas Protegidas — Región Andina (datos sintéticos)', fontweight='bold')
    ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')
    ax.legend(loc='lower right')
    plt.tight_layout(); plt.show()

    # Resumen ejecutivo por iniciativa
    if not traslapes.empty:
        print('\nResumen por iniciativa REDD+:')
        resumen = traslapes.groupby('id_redd').agg(
            n_areas_protegidas=('id_ap', 'count'),
            ha_traslape_total=('ha_traslape', 'sum'),
            pct_max_cubierto=('pct_of_id_redd', 'max'),
        ).round(2)
        print(resumen)

### Interpretación

- **`pct_of_id_redd`** → fracción de la iniciativa cubierta por figura de conservación
- **`pct_of_id_ap`** → fracción del área protegida cubierta por la iniciativa

> **Regla práctica:** Si `pct_of_id_redd > 50%`, la iniciativa se superpone mayoritariamente
> con una figura de conservación existente. En ese caso verificar con el registro RENARE y la
> categoría de la figura — un **PNN** tiene restricción total de nuevas actividades; una
> **Reserva Forestal** puede tener compatibilidad parcial con REDD+.

---

## Caso 2 — Estadísticas zonales: emisiones GEI por departamento

### Contexto

Cuando tenemos un **raster continuo** (densidad de emisiones en tCO₂e/km², deforestación en ha,
NDVI) necesitamos **agregar esos valores por unidad político-administrativa** para reportes de NDC,
seguimiento IPCC o análisis comparativos departamentales.

Equivale a la función *Zonal Statistics* de QGIS o *Zonal Statistics as Table* de ArcGIS.

### Fuentes de datos raster colombianas

| Variable | Fuente | Resolución | Acceso |
|---|---|---|---|
| Deforestación | SMByC / IDEAM | 30m (Landsat) | https://smbyc.ideam.gov.co |
| Cobertura tierra | CORINE Land Cover Colombia | 30m | IDEAM / SIAC |
| NDVI / biomasa | Sentinel-2 (GEE) | 10m | earthengine.google.com |
| Temperatura superficial | MODIS LST | 1km | NASA Earthdata |

In [ ]:
import tempfile, os

if HAS_RASTERIO and HAS_GPD:
    # Raster sintético: densidad de emisiones (tCO₂e/km²)
    lon_min, lon_max, lat_min, lat_max = -75.5, -72.5, 3.5, 7.0
    rows, cols = 70, 60

    rng = np.random.default_rng(42)
    x_grad = np.linspace(1.5, 0.5, cols)   # más emisiones al occidente
    y_grad = np.linspace(0.8, 1.2, rows)
    emisiones = (np.outer(y_grad, x_grad) * 45 + rng.normal(0, 5, (rows, cols))).clip(0)

    transform = from_bounds(lon_min, lat_min, lon_max, lat_max, cols, rows)
    tmp = tempfile.NamedTemporaryFile(suffix='.tif', delete=False)
    with rasterio.open(tmp.name, 'w', driver='GTiff',
                       height=rows, width=cols, count=1, dtype='float32',
                       crs='EPSG:4326', transform=transform, nodata=-9999) as dst:
        dst.write(emisiones.astype('float32'), 1)

    print(f'Raster sintético: {rows}×{cols} px | {emisiones.min():.1f}–{emisiones.max():.1f} tCO₂e/km²')

    # Polígonos de departamentos (sintéticos, coordenadas reales de Colombia)
    deptos = gpd.GeoDataFrame(
        {
            'COD_DANE':    ['25', '15', '73', '68'],
            'DEPARTAMENTO': ['Cundinamarca', 'Boyacá', 'Tolima', 'Santander'],
            'geometry': [
                box(-74.8, 3.7, -73.5, 5.1),
                box(-73.5, 5.0, -72.5, 7.0),
                box(-75.5, 3.5, -74.5, 5.3),
                box(-73.5, 5.5, -72.5, 7.0),
            ],
        },
        crs='EPSG:4326',
    )
    print(f'Departamentos: {len(deptos)}')
else:
    print('Requiere rasterio + geopandas: pip install estadistica-ambiental[spatial]')

In [ ]:
if HAS_RASTERIO and HAS_GPD:
    result_zonal = zonal_statistics(
        raster_path=tmp.name,
        zones_gdf=deptos,
        zone_id_col='COD_DANE',
        stats=['mean', 'sum', 'std', 'min', 'max'],
    )

    # Emisiones totales estimadas (sum × área pixel ≈ 30 km² — aprox ilustrativa)
    result_zonal['MtCO2e_total'] = (result_zonal['sum'] * 0.030).round(2)

    print('Estadísticas zonales de emisiones GEI (tCO₂e/km²):\n')
    cols_show = ['DEPARTAMENTO', 'mean', 'std', 'min', 'max', 'MtCO2e_total']
    print(result_zonal[cols_show].round(2).to_string(index=False))

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

    result_zonal.plot(column='mean', ax=ax1, cmap='YlOrRd', legend=True,
                      legend_kwds={'label': 'tCO₂e/km² (media)'})
    ax1.set_title('Densidad media de emisiones', fontweight='bold')

    result_zonal.plot(column='MtCO2e_total', ax=ax2, cmap='Reds', legend=True,
                      legend_kwds={'label': 'MtCO₂e totales'})
    ax2.set_title('Emisiones totales estimadas', fontweight='bold')

    for ax in [ax1, ax2]:
        for _, row in result_zonal.iterrows():
            ax.annotate(row['DEPARTAMENTO'],
                        (row.geometry.centroid.x, row.geometry.centroid.y),
                        ha='center', fontsize=8)

    fig.suptitle('Estadísticas zonales — Emisiones GEI por departamento (datos sintéticos)', fontweight='bold')
    plt.tight_layout(); plt.show()

    os.unlink(tmp.name)

---

## Caso 3 — Interpolación espacial: estaciones IDEAM

### ¿Cuándo usar cada método?

| Método | Cuándo usarlo | Limitación |
|---|---|---|
| **IDW** | Exploración rápida, distribución uniforme de puntos | No modela autocorrelación espacial |
| **Kriging ordinario** | Variable estacionaria, variograma ajustable | Asume media global constante |
| **Kriging universal** | Variable con gradiente claro (temperatura ~ elevación) | Requiere drift conocido |

**Caso típico:** estimar temperatura media anual en municipios sin estación directa,
usando estaciones IDEAM de la zona andina como puntos de control.

In [ ]:
# Estaciones IDEAM sintéticas — Cundinamarca y Boyacá
estaciones = pd.DataFrame({
    'estacion':    ['BOG-001','BOG-002','BYC-001','BYC-002','BYC-003',
                    'CND-001','CND-002','CND-003','CND-004','CND-005'],
    'lat':         [4.60, 4.85, 5.30, 5.55, 5.80, 4.20, 4.45, 4.70, 4.30, 5.10],
    'lon':         [-74.10,-73.95,-73.50,-73.30,-73.10,-74.40,-74.25,-74.05,-73.85,-73.65],
    # Temperatura media anual (°C) — gradiente altitudinal Andes colombianos
    'temp_media':  [13.5, 14.2, 12.8, 11.5, 10.2, 16.8, 15.3, 14.7, 15.9, 13.1],
    'elevacion_m': [2600, 2480, 2750, 3100, 3400, 1800, 2100, 2300, 1950, 2700],
})

print(f'Estaciones: {len(estaciones)}')
print(f'Temperatura: {estaciones["temp_media"].min()}–{estaciones["temp_media"].max()} °C')
print(f'Elevación:   {estaciones["elevacion_m"].min()}–{estaciones["elevacion_m"].max()} m\n')
print(estaciones[['estacion', 'lat', 'lon', 'temp_media', 'elevacion_m']])

In [ ]:
# Grilla de interpolación — región andina colombiana
grid_lat, grid_lon = np.meshgrid(
    np.linspace(4.0, 6.0, 80),
    np.linspace(-74.6, -72.9, 80),
    indexing='ij',
)

# IDW (siempre disponible, sin dependencias opcionales)
z_idw = idw(estaciones, 'lat', 'lon', 'temp_media', grid_lat, grid_lon, power=2.0)
print(f'IDW              — rango: {z_idw.min():.2f}–{z_idw.max():.2f} °C')

# Kriging ordinario (requiere pykrige)
if HAS_PYKRIGE:
    z_ok, ss_ok = ordinary_kriging(estaciones, 'lat', 'lon', 'temp_media',
                                    grid_lat, grid_lon, variogram_model='spherical')
    print(f'Kriging ordinario — rango: {z_ok.min():.2f}–{z_ok.max():.2f} °C')

    z_uk, ss_uk = universal_kriging(estaciones, 'lat', 'lon', 'temp_media',
                                     grid_lat, grid_lon,
                                     variogram_model='linear', drift_order=1)
    print(f'Kriging universal  — rango: {z_uk.min():.2f}–{z_uk.max():.2f} °C')
else:
    print('pykrige no instalado — solo se muestra IDW')

In [ ]:
n_plots = 3 if HAS_PYKRIGE else 1
fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 6))
if n_plots == 1:
    axes = [axes]

vmin = estaciones['temp_media'].min() - 0.5
vmax = estaciones['temp_media'].max() + 0.5

capas = [(z_idw, 'IDW (power=2)')]
if HAS_PYKRIGE:
    capas += [(z_ok, 'Kriging Ordinario\n(esférico)'), (z_uk, 'Kriging Universal\n(drift lineal)')]

for ax, (z, titulo) in zip(axes, capas):
    img = ax.pcolormesh(grid_lon, grid_lat, z, cmap='RdYlBu_r',
                        vmin=vmin, vmax=vmax, shading='auto')
    plt.colorbar(img, ax=ax, label='Temperatura media anual (°C)')
    ax.scatter(estaciones['lon'], estaciones['lat'],
               c=estaciones['temp_media'], cmap='RdYlBu_r',
               vmin=vmin, vmax=vmax, s=90, edgecolors='k', linewidth=0.8, zorder=5)
    for _, row in estaciones.iterrows():
        ax.annotate(f"{row['temp_media']}°C",
                    (row['lon'], row['lat']), textcoords='offset points',
                    xytext=(4, 4), fontsize=6)
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Longitud'); ax.set_ylabel('Latitud')

fig.suptitle('Temperatura media anual — Estaciones IDEAM (sintéticas) | Región Andina',
             fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

# Mapa de incertidumbre del Kriging
if HAS_PYKRIGE:
    fig, ax = plt.subplots(figsize=(7, 6))
    img = ax.pcolormesh(grid_lon, grid_lat, np.sqrt(ss_ok), cmap='YlOrRd', shading='auto')
    plt.colorbar(img, ax=ax, label='Desviación estándar estimación (°C)')
    ax.scatter(estaciones['lon'], estaciones['lat'], c='black', s=50, zorder=5, label='Estaciones')
    ax.set_title('Incertidumbre Kriging Ordinario\n(mayor lejos de las estaciones)', fontweight='bold')
    ax.legend()
    plt.tight_layout(); plt.show()

### ¿Cuándo preferir Kriging Universal?

> En Colombia la temperatura siempre tiene tendencia con la elevación (~−6.5°C por 1000m).
> El Kriging Ordinario ignora ese gradiente — produce estimaciones incorrectas en zonas de
> alta variación altitudinal. Usar **Kriging Universal con `drift_order=1`** como punto de
> partida para cualquier variable con gradiente espacial conocido.
>
> Si el variograma experimental no estabiliza (sigue creciendo), la variable no es estacionaria
> → usar Universal Kriging o transformar (diferencias) antes de modelar.

---

## Caso 4 — Autocorrelación espacial: hotspots de deforestación

### ¿Por qué verificar autocorrelación antes de modelar?

Los modelos OLS asumen independencia entre observaciones. En datos espaciales esto casi nunca
se cumple (Ley de Tobler). Usar regresión sin corregir produce **errores estándar subestimados**
y tests de hipótesis inválidos.

**Flujo estándar:**

```
morans_i()      → ¿existe autocorrelación global significativa?
geary_c()       → confirmación con C de Geary (más sensible a disimilitud local)
local_morans_i() → ¿dónde están los clusters HH/LL/HL/LH?
getis_ord_g()   → hotspots y coldspots estadísticamente significativos (G*)
```

Si la autocorrelación es significativa → usar **GWR** (coeficientes locales) o
**SAR/SEM** (modelos de autocorrelación espacial global) en lugar de OLS.

In [ ]:
if not (HAS_GPD and HAS_ESDA):
    print('Requiere: pip install estadistica-ambiental[spatial]')
else:
    rng = np.random.default_rng(123)

    # GeoDataFrame: tasa de deforestación por municipio — cuadrícula 4×4 (Caquetá sintético)
    geoms = [box(i, j, i+1, j+1) for j in range(4) for i in range(4)]

    # Patrón espacial inducido: cluster de alta deforestación en esquina SO
    deforest = []
    for g in geoms:
        cx, cy = g.centroid.x, g.centroid.y
        base = 8.5 if (cx < 2 and cy < 2) else 2.0
        deforest.append(max(0, base + rng.normal(0, 0.4)))

    municipios = gpd.GeoDataFrame({
        'municipio':    [f'MUN-{i:02d}' for i in range(16)],
        'deforest_pct': deforest,
        'geometry':     geoms,
    }, crs='EPSG:4326')

    print(f'Municipios: {len(municipios)}')
    print(f'Deforestación media:  {municipios["deforest_pct"].mean():.2f}%')
    print(f'Deforestación máxima: {municipios["deforest_pct"].max():.2f}%')

In [ ]:
if HAS_GPD and HAS_ESDA:
    # I de Moran global
    mi = morans_i(municipios, 'deforest_pct', weight_type='queen')
    print('=== I de Moran Global ===')
    for k, v in mi.items():
        print(f'  {k:15s}: {v}')

    print()

    # C de Geary (complemento — más sensible a variación local)
    gc = geary_c(municipios, 'deforest_pct', weight_type='queen')
    print('=== C de Geary ===')
    for k, v in gc.items():
        print(f'  {k:15s}: {v}')

In [ ]:
if HAS_GPD and HAS_ESDA:
    gdf_g   = getis_ord_g(municipios, 'deforest_pct', weight_type='queen')
    gdf_lisa = local_morans_i(municipios, 'deforest_pct', weight_type='queen')
    lisa_labels = {1: 'HH (hotspot)', 2: 'LH', 3: 'LL (coldspot)', 4: 'HL'}
    gdf_lisa['lisa_label'] = gdf_lisa['lisa_q'].map(lisa_labels)

    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    # Mapa coroplético — tasa de deforestación
    municipios.plot(column='deforest_pct', cmap='YlOrRd', legend=True, ax=axes[0],
                    legend_kwds={'label': '% deforestado'})
    axes[0].set_title('Tasa de deforestación (%)', fontweight='bold')

    # G* Getis-Ord — hotspots
    for tipo, color in [('hot','#e74c3c'), ('cold','#3498db'), ('ns','#bdc3c7')]:
        subset = gdf_g[gdf_g['hotspot'] == tipo]
        if not subset.empty:
            subset.plot(color=color, ax=axes[1], label=tipo, edgecolor='white', linewidth=0.5)
    axes[1].set_title('Hotspots G* de Getis-Ord\nrojo=hot · azul=cold · gris=ns', fontweight='bold')
    axes[1].legend(fontsize=8)

    # LISA clusters
    colores = {'HH (hotspot)': '#d73027', 'LH': '#fc8d59',
               'LL (coldspot)': '#4575b4', 'HL': '#91bfdb'}
    for label, color in colores.items():
        sub = gdf_lisa[gdf_lisa['lisa_label'] == label]
        if not sub.empty:
            sub.plot(color=color, ax=axes[2], label=label, edgecolor='white', linewidth=0.5)
    gdf_lisa[~gdf_lisa['lisa_sig']].plot(color='#bdc3c7', ax=axes[2],
                                          label='No significativo', edgecolor='white')
    axes[2].set_title('LISA — Moran Local\nclusters y outliers espaciales', fontweight='bold')
    axes[2].legend(fontsize=7)

    plt.tight_layout(); plt.show()

    # Municipios con hotspot significativo
    hot = gdf_g[gdf_g['hotspot'] == 'hot'][['municipio', 'deforest_pct', 'g_z', 'g_p']]
    if not hot.empty:
        print(f'\nMunicipios con hotspot significativo (G*): {len(hot)}')
        print(hot.round(3).to_string(index=False))

---

## Árbol de decisión metodológico

```
¿Tengo datos con componente espacial?
│
├─► Puntos (estaciones de monitoreo)
│    ├─► ¿Quiero estimar valores en zonas sin monitoreo?
│    │    ├─► Distribución uniforme, exploración rápida → IDW
│    │    ├─► Variable estacionaria, variograma ajustable → Kriging Ordinario
│    │    └─► Gradiente altitudinal / tendencia conocida → Kriging Universal
│    └─► ¿Quiero detectar agrupamientos?
│         └─► morans_i() → getis_ord_g() → local_morans_i() LISA
│
├─► Polígonos (iniciativas, áreas protegidas, municipios)
│    ├─► ¿Traslapes entre capas? → intersection_area()  ← Caso 1
│    ├─► ¿Agregar raster por zona? → zonal_statistics() ← Caso 2
│    └─► ¿Modelar variación espacial de un fenómeno?
│         └─► GWR (Regresión Geográficamente Ponderada) — librería: mgwr
│
└─► Raster (Sentinel-2, MODIS, SRTM)
     ├─► Resolución insuficiente (<10m necesario) → SEN2SR (requiere GPU)
     ├─► Agregar por zona → zonal_statistics()
     └─► Análisis temporal → xarray + load_netcdf_spatial()
```

---

## Checklist antes de cualquier análisis espacial

- [ ] **CRS documentado:** ¿todas las capas están en el mismo sistema de referencia?
- [ ] **Proyección métrica para áreas:** CTM12 (EPSG:9377) — no WGS84 — para `intersection_area()`
- [ ] **Topología válida:** `load_vector()` auto-corrige con `buffer(0)`, verificar el log
- [ ] **NoData identificado:** rasters Copernicus/MODIS siempre tienen zonas sin datos
- [ ] **Escala apropiada:** resolución del raster compatible con tamaño de los polígonos
- [ ] **Autocorrelación evaluada** antes de OLS o modelos que asumen independencia

---

## Integración con el resto del repositorio

```
spatial/io.py           → load_vector(), load_raster()       entrada de datos
spatial/projections.py  → reproject(), clip_to_colombia()    normalización CRS
spatial/analysis.py     → intersection_area()                Caso 1: traslapes
                        → zonal_statistics()                 Caso 2: por municipio
spatial/interpolation.py → idw(), ordinary_kriging()         Caso 3: mapas continuos
                         → universal_kriging()               con gradiente espacial
spatial/autocorrelation.py → morans_i(), geary_c()           Caso 4: dependencia
                           → getis_ord_g()                   hotspots deforestación
spatial/viz.py          → map_stations(), choropleth_map()   visualización folium

↓ Alimenta a:
  evaluation/metrics.py       métricas por zona (NSE/KGE para cuencas)
  inference/intervals.py      exceedance_report() con dimensión espacial
  reporting/compliance_report.py  cumplimiento normativo por municipio
```

*Retroalimentación: Juan (SIG — Ministerio de Ambiente y Desarrollo Sostenible, 2026).*  
*Metodología: estándares internacionales (OGC, ISO 19152 LADM) con implementación de referencia para Colombia.*

## 7b. Guardrails y supuestos metodológicos
<!-- GUARDRAILS: geoespacial -->

> **Antes de publicar resultados**, verificar que se cumplen los supuestos clave del flujo. Esta sección lista los más comunes y los específicos de la línea.

### Supuestos comunes (todas las líneas)

- **Estacionariedad (ADR-004):** ADF + KPSS deben coincidir antes de aplicar ARIMA. Si discrepan, diferenciar conservadoramente o usar modelos no-ARIMA (Prophet, ML).
- **Outliers (ADR-002):** los picos ambientales son señal real (eventos, episodios). No aplicar clipping automático — sólo `preprocessing/outliers.py` opt-in y documentado.
- **Métrica primaria (ADR-003):** RMSLE NO en variables que pueden ser negativas o cero. Usar MAE + sMAPE (o NSE / KGE en hidrología) como default.
- **Tamaño muestral mínimo:** ARIMA requiere ≥ 36 observaciones; STL anual con datos diarios, ≥ 2 ciclos completos.
- **Residuos (post-fit):** verificar normalidad (Jarque-Bera) e independencia (Ljung-Box, lag = 12). Residuos correlacionados → modelo subespecificado.
- **Walk-forward con gap (BP-1):** series con ACF ≥ 0.7 inflan R². Usar `gap ≥ horizonte` en `walk_forward()`.
- **Normas oficiales:** usar `config.NORMA_*` y `config.*_THRESHOLDS` — nunca umbrales hardcodeados en el notebook (ADR-005).

### Supuestos específicos — Geoespacial / interpolación

- **Autocorrelación espacial:** Moran's I antes de modelar regresión sobre datos geográficos.
- **Kriging:** verificar variograma teórico (esférico / exponencial / gaussiano) — el ajuste por OLS al variograma empírico no es trivial.
- **CRS coherente:** EPSG:4326 (WGS84) para análisis general, EPSG:3116 (MAGNA-SIRGAS Origen Bogotá) para Colombia continental.

### Antes de presentar a la autoridad ambiental

- Reportar intervalos de confianza, no sólo el punto estimado.
- Documentar el período de los datos, los gaps y el método de imputación usado.
- Registrar decisiones metodológicas no triviales en `docs/decisiones.md` (ADRs).
